# encoder
# decoder
# train
# generate

In [2]:
import torch
import torch.nn as nn

In [3]:
chars = "0123456789-/"
special_tokens = ["<SOS>", "<EOS>"]

itos = special_tokens + list(chars)
stoi = {ch: i for i, ch in enumerate(itos)}

SOS_token = stoi["<SOS>"]
EOS_token = stoi["<EOS>"]

vocab_size = len(itos)

print(itos)
print(stoi)

['<SOS>', '<EOS>', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '-', '/']
{'<SOS>': 0, '<EOS>': 1, '0': 2, '1': 3, '2': 4, '3': 5, '4': 6, '5': 7, '6': 8, '7': 9, '8': 10, '9': 11, '-': 12, '/': 13}


In [4]:
class EncoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size)
        self.dropout = nn.Dropout(0.1)

        # 你来写 embedding
        # 你来写 rnn

    def forward(self, input):
        # 你来写一次前向传播
        input = self.embedding(input)
        input = self.dropout(input)
        input = input.unsqueeze(1)
        output, hidden = self.rnn(input)
        return output, hidden


In [5]:
import random
def make_sample():
    ep = 100
    inputs = []
    targets = []
    for i in range(ep):
        year = random.randint(1950, 2050)
        day = random.randint(1, 28)
        month = random.randint(1, 12)
        input = f"{year}-{month}-{day}"
        inputs.append(input)
        target = f"{day}/{month}/{year}"
        targets.append(target)
    return inputs, targets

inputs, targets = make_sample()
print(inputs[0:10])
print(targets[0:10])


['2041-2-16', '1956-10-10', '1966-2-20', '1993-4-15', '2032-10-17', '1978-10-14', '1961-12-25', '2039-10-1', '2011-10-11', '1998-10-17']
['16/2/2041', '10/10/1956', '20/2/1966', '15/4/1993', '17/10/2032', '14/10/1978', '25/12/1961', '1/10/2039', '11/10/2011', '17/10/1998']


In [6]:
encoder = EncoderRNN(vocab_size, 5)
input_tensor = torch.tensor([stoi[ch] for ch in inputs[0]], dtype=torch.long)
print(input_tensor)
output,hidden = encoder.forward(input_tensor)
print(output,hidden)

tensor([ 4,  2,  6,  3, 12,  4, 12,  3,  8])
tensor([[[ 0.0499,  0.5427,  0.6613, -0.7457,  0.6570]],

        [[-0.6073,  0.3499, -0.0796, -0.8160, -0.0397]],

        [[-0.2684, -0.2992,  0.4019, -0.6238, -0.0240]],

        [[ 0.7929, -0.2359,  0.9764, -0.3137, -0.2481]],

        [[-0.8201, -0.5523,  0.7224, -0.4888, -0.4404]],

        [[-0.4745, -0.1430,  0.8659, -0.6083,  0.7741]],

        [[-0.8602, -0.2409,  0.3604, -0.7951, -0.0045]],

        [[ 0.7397, -0.2283,  0.9712, -0.3816, -0.1102]],

        [[-0.6419, -0.3650,  0.2611, -0.3330, -0.7290]]],
       grad_fn=<StackBackward0>) tensor([[[-0.6419, -0.3650,  0.2611, -0.3330, -0.7290]]],
       grad_fn=<StackBackward0>)


In [7]:
print(output.shape)
print(hidden.shape)

torch.Size([9, 1, 5])
torch.Size([1, 1, 5])


In [ ]:
class DecoderRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_output, target_tensor):
        # encoder_output是encoder输出的hiddeng向量
        # target_tensor是decoder输出的向量
        last_target_tensor #记住上一次生成的词元id
        hidden = encoder_output
        input = []
        input.append(SOS_token)
        outputs = []
        for i in range(12):
            if target_tensor is not None:
                input.append(target_tensor[i:i+1])
           
            embedded = self.embedding(input)
            rnn_output, hidden = nn.rnn(embedded, hidden)
            decode_ouput = self.out(rnn_output)
            predict = decode_ouput.argmax(dim = -1, keepdim=True)
            outputs.append(predict)
            last_target_tensor = self.embedding(predict)
            if target_tensor is None:
                input = predict
        return torch.cat(outputs, dim=1)  # (batch, max_len)
        
    def forward_step(self, hidden, input_token):
        embedded = self.embedding(input_token)
        rnn_output, hidden = self.rnn(embedded, hidden)
        output = self.out(hidden[-1])
        predictIdx = output.argmax(dim = -1, keepdim=True)
        return predictIdx

    

In [ ]:
hidden_size = 5
decoder = DecoderRNN(vocab_size, hidden_size, vocab_size)


In [11]:
print(decoder.out.shape)

AttributeError: 'Linear' object has no attribute 'shape'